# Environment Setup

**Goal:** Understand the Miniconda/conda workflow for managing scientific Python environments, and when to reach for `venv` or `uv` instead.

**We will cover**
- Installing Miniconda.
- Creating and managing conda environments, then layering in `pip` or `uv` for edge cases.
- Capturing environments (`environment.yml`, `requirements.txt`, `uv` lock files) and registering kernels.
- When it still makes sense to reach for `python -m venv` or `uv` **after** you have Miniconda.

**[Anaconda's page on Getting Started](https://www.anaconda.com/docs/getting-started/getting-started)**


## Why standardize on Miniconda for this course
- **Reproducibility & parity:** Everyone gets the exact interpreter + compiled deps (`numpy`, `pandas`, `pytorch`, GDAL, etc.) from conda-forge regardless of OS updates.
- **Apple Silicon support:** The Miniconda `MacOSX-arm64` installer ships universal OpenBLAS, SSL, and other native libs tuned for the M-series chips, so no Xcode/CLT compilers are required.

## Install Miniconda on Apple Silicon (shell workflow)

Check [here](https://www.anaconda.com/docs/getting-started/miniconda/install) for the latest Miniconda installer links and instructions for other platforms.

Run this command in Terminal:

```bash
curl -O https://repo.anaconda.com/miniconda/Miniconda3-latest-MacOSX-arm64.sh
```
The above `curl` command downloads the latest Minicondaarm64 installer.

```bash
bash ~/Miniconda3-latest-MacOSX-arm64.sh
```
This command runs the installer script. Accept the license terms, and install to the default location (`/Users/yourusername/miniconda3`).

Close and re-open your terminal window for the installation to fully take effect, or use the following command to refresh the terminal:

```bash
source ~/.zshrc
```

Check that conda is installed correctly:

```bash
conda info
```
You should see output showing conda's configuration and version information.

> **Danger:** Always download the `MacOSX-arm64` build for Apple Silicon. Installing an Intel (`x86_64`) build on M-series hardware forces Rosetta emulation and misses native packages.

In [ ]:
# We can check from here by using ! commands in a Jupyter notebook:
!conda info

## Conda workflow cheat sheet (what we expect you to run)

- **Create a project env:** `conda create -n ds-env python=3.13 numpy pandas -c conda-forge`
- **Activate / deactivate:** `conda activate ds-env` and `conda deactivate`
- **Install heavy packages first:** prefer `conda install -c conda-forge ...` for anything with native libs (PyTorch, GDAL, arrow, etc.).
- **Use pip inside the env when needed:** `conda run -n ds-env python -m pip install streamlit` keeps the base environment clean.
- **Remove / clean:** `conda env remove -n ds-env`.

Typical environment bootstrap:

```bash
conda activate ds-env
conda install ipykernel jupyterlab -c conda-forge
python -m ipykernel install --user --name ds-env --display-name "Python (ds-env)"
```

This registers the kernel so JupyterLab/VS Code can find the interpreter.


## Capture and share environments

### Conda (`environment.yml`)
```yaml
name: ds-env
channels:
  - conda-forge
dependencies:
  - python=3.13
  - numpy>=2.1
  - pandas
  - pip
  - pip:
      - pytorch
```
Export/update with `conda env export -n ds-env > environment.yml` and recreate elsewhere via `conda env create -f environment.yml`.

### Pip or uv extras inside the same env
- When you need a PyPI-only wheel, install it **after activating** the conda env (`python -m pip install rich`). Capture those additions with `pip freeze > requirements.txt` or the `pip:` section above.
- `uv` can target the conda-managed interpreter for lightning-fast wheel resolution:
  ```bash
  uv pip install -r requirements.txt
  uv pip compile requirements.in -o requirements.lock
  uv pip sync requirements.lock
  ```
  `uv` hashes each wheel, so CI or collaborators can verify supply chain integrity.

### Kernel registration reminders
1. Install `ipykernel` in the active env (`conda install ipykernel` or `python -m pip install ipykernel`).
2. Register: `python -m ipykernel install --user --name ds-env --display-name "Python (ds-env)"`.
3. VS Code → `Python: Select Interpreter`, JupyterLab launcher, or `jupyter kernelspec list` should now show `ds-env`.
4. After rebuilding an env, rerun the registration so the kernelspec points to the new interpreter path.


## Where `python -m venv` or `uv` still help (after Conda)

| Tool | Reach for it when… | Highlights | Trade-offs |
| --- | --- | --- | --- |
| **Conda / Mamba** | Default for this course; you need pre-built scientific stacks or multiple Python versions | Solves native deps, `conda-forge` breadth, easy env export | Larger installs, solver slower than `uv` |
| **`python -m venv`** | You must match stock CPython exactly (packaging tests, minimal images) | Bundled with Python, zero extra tooling | Wheels only; native deps require compilers on macOS |
| **`uv`** | You want ultra-fast wheel installs or deterministic lock files | Resolver is extremely fast, integrates with `requirements.txt` | No native builds—relies on PyPI wheels provided by maintainers |

Quick references (only after Miniconda is installed):

```bash
# Built-in venv
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt

# uv using the conda-managed interpreter
uv venv .venv
source .venv/bin/activate
uv pip install numpy pandas jupyterlab ipykernel
uv pip freeze > requirements.txt
```

Even when experimenting with `venv` or `uv`, keep your primary conda env for day-to-day notebook work so everyone stays on the same stack.


This next cell 